# Containership Routing with a Route-Duration Limit: Tanjung Perak Hub-and-Spoke Network
**Revised version** of `VRP_MTZ_1Hub16Spoke_TMax_Gurobi_ok.ipynb`.

Main changes:
1. **Two-index formulation** `x[i,j]` (instead of `x[i,j,k]`). Identical ships create symmetric duplicate solutions in the three-index model, which is why the old run stalled at a 7.48% gap. Fleet size is limited by the number of hub departures, `<= K`.
2. **Correct capacitated MTZ**: `u_j >= u_i + d_j - Q(1 - x_ij)`, with `u` in tons.
3. **Arrival-time variables** `t_i` with port **service time** (Sheet2 of the Excel file) counted inside the `Tmax` limit.
4. **Simultaneous pickup and delivery** mode (`pd`) using the *Supply* column.
5. Independent solution check, distance-matrix quality checks, a **regression test**, and a **Tmax sensitivity analysis**.

> **Note on time bounds.** The distance matrix violates the triangle inequality, so the earliest and latest arrival bounds use *shortest-path* distances (Floyd-Warshall), not direct arcs. An earlier draft of this notebook used direct arcs, which wrongly excluded feasible routes (it reported 9,727 nm as optimal for Tmax = 350 although a 9,708 nm solution exists). Section 5 contains a regression test that guards against this.

## 0. Settings (edit here)

In [ ]:
EXCEL_PATH = r"D:\00 - DISSERTATION\03 - LSNW Collaboration\DATA Processing\Test Data Hub and Spok.xlsx"   # path to your Excel file
import os
os.environ["GRB_LICENSE_FILE"] = r"C:\gurobi1202\win64\gurobi.lic"   # path to your Gurobi license file (must run before the first model is built)
TIME_LIMIT = 600                              # seconds per run
RUN_SWEEP  = False                            # True = run the sensitivity table (slow)

## 1. Installation and imports

In [ ]:
# Run once if the packages are not installed yet:
# %pip install gurobipy pandas openpyxl folium matplotlib

In [ ]:
import math
from dataclasses import dataclass, field

import pandas as pd
import gurobipy as gp
from gurobipy import GRB

## 2. Data: distances, demand, supply, service time
`load_instance` also checks symmetry and the **triangle inequality** of the distance matrix. A violation means some distance is geographically implausible and should be verified against an official distance table.

In [ ]:
PORT_NAMES = {
    1: "Tanjung Perak", 2: "Balikpapan", 3: "Banjarmasin", 4: "Tarakan",
    5: "Samarinda", 6: "Palu", 7: "Gorontalo", 8: "Luwuk", 9: "Tanjung Redep",
    10: "Sampit", 11: "Kendari", 12: "Bau-Bau", 13: "Teluk Bayur",
    14: "Pekanbaru", 15: "Kota Baru", 16: "Batu Ampar",
}


@dataclass
class Instance:
    ports: list                      # [1..n], port 1 = hub
    dist: dict                       # (i, j) -> nautical miles
    demand: dict                     # i -> tons delivered hub -> i
    supply: dict                     # i -> tons picked up i -> hub
    service: dict                    # i -> port service (loading/unloading) time in hours
    Q: float = 870.0                 # ship capacity (tons)
    speed: float = 12.0              # knots
    names: dict = field(default_factory=lambda: PORT_NAMES)

    @property
    def hub(self):
        return self.ports[0]

    @property
    def spokes(self):
        return self.ports[1:]

    def sail(self, i, j):
        """Sailing time (hours) on the direct arc i -> j."""
        return self.dist[i, j] / self.speed


def load_instance(excel_path, Q=870.0, speed=12.0):
    """Read Sheet1 (distance matrix) and Sheet2 (demand, supply, service time)."""
    dm = pd.read_excel(excel_path, sheet_name="Sheet1")
    dm = dm.iloc[:, 1:].apply(pd.to_numeric)
    n = dm.shape[0]
    ports = list(range(1, n + 1))
    dist = {(i, j): float(dm.iloc[i - 1, j - 1]) for i in ports for j in ports if i != j}

    # Data quality checks: asymmetry and triangle-inequality violations flag distances to verify.
    asym = [(i, j) for (i, j) in dist if abs(dist[i, j] - dist[j, i]) > 1e-6]
    tri = [(i, k, j) for i in ports for k in ports for j in ports
           if len({i, j, k}) == 3 and dist[i, j] > dist[i, k] + dist[k, j] + 1e-6]
    if asym:
        print(f"[warning] distance matrix is asymmetric for {len(asym)} pairs, e.g. {asym[:3]}")
    if tri:
        print(f"[warning] triangle inequality violated in {len(tri)} triplets, e.g. {tri[:3]}")

    s2 = pd.read_excel(excel_path, sheet_name="Sheet2")
    s2.columns = [c.strip() for c in s2.columns]
    s2 = s2.set_index("Port")
    demand = {i: float(s2.loc[i, "Demand"]) for i in ports}
    supply = {i: float(s2.loc[i, "Supply"]) for i in ports}
    service = {i: float(s2.loc[i, "Service Time_Hours"]) for i in ports}
    # The hub's own demand/supply is not ship cargo, and hub service time is ignored.
    demand[ports[0]] = 0.0
    supply[ports[0]] = 0.0
    service[ports[0]] = 0.0
    return Instance(ports, dist, demand, supply, service, Q, speed)


## 3. Optimization model
`mode='delivery'` is the CVRP with a route-duration limit. `mode='pd'` adds simultaneous pickup and delivery.

In [ ]:
def build_model(inst: Instance, tmax: float, kmax: int, mode="delivery",
                use_service=True, kmin=0, time_limit=600, mip_gap=0.0, verbose=False):
    """
    mode = "delivery": CVRP with a route-duration limit. Cargo is tracked by u_i (capacitated MTZ).
    mode = "pd"      : VRP with simultaneous pickup and delivery. Cargo is tracked by arc flows
                       D_ij (delivery cargo still on board) and P_ij (pickup cargo already on board).
    In both modes, time is tracked by t_i = service start time at port i (hours since leaving the hub).
    """
    H = inst.hub
    S = inst.spokes
    N = inst.ports
    A = [(i, j) for i in N for j in N if i != j]
    svc = inst.service if use_service else {i: 0.0 for i in N}
    Q = inst.Q

    # Shortest-path distances (Floyd-Warshall). Node-level time bounds MUST use shortest paths,
    # not direct distances: the matrix violates the triangle inequality, so a route through
    # another port can be shorter than the direct arc. Direct-arc bounds would wrongly cut off
    # feasible solutions (and make "optimal" results too pessimistic).
    sp = {(i, j): inst.dist[i, j] for (i, j) in A}
    for k in N:
        for i in N:
            for j in N:
                if len({i, j, k}) == 3 and sp[i, k] + sp[k, j] < sp[i, j]:
                    sp[i, j] = sp[i, k] + sp[k, j]

    m = gp.Model(f"DCVRP_{mode}")
    m.Params.OutputFlag = 1 if verbose else 0
    m.Params.TimeLimit = time_limit
    m.Params.MIPGap = mip_gap

    x = m.addVars(A, vtype=GRB.BINARY, name="x")
    t = m.addVars(S, lb={i: sp[H, i] / inst.speed for i in S},
                  ub={i: tmax - svc[i] - sp[i, H] / inst.speed for i in S}, name="t")

    # (1) objective: total sailing distance
    m.setObjective(gp.quicksum(inst.dist[i, j] * x[i, j] for (i, j) in A), GRB.MINIMIZE)

    # (2) every spoke is visited exactly once (one arc in, one arc out)
    m.addConstrs((x.sum("*", j) == 1 for j in S), name="in")
    m.addConstrs((x.sum(j, "*") == 1 for j in S), name="out")

    # (3) number of routes = number of departures from the hub, bounded by kmin..kmax
    m.addConstr(x.sum(H, "*") <= kmax, name="fleet_max")
    m.addConstr(x.sum(H, "*") >= kmin, name="fleet_min")
    m.addConstr(x.sum(H, "*") == x.sum("*", H), name="hub_balance")

    # (4) time propagation along spoke -> spoke arcs:
    #     t_j >= t_i + s_i + sail_ij if x_ij = 1. Increasing time also eliminates subtours.
    for (i, j) in A:
        if i != H and j != H:
            M = tmax + svc[i] + inst.sail(i, j)
            m.addConstr(t[j] >= t[i] + svc[i] + inst.sail(i, j) - M * (1 - x[i, j]), name=f"time[{i},{j}]")

    # (4b) first and last arc of each route: the first port cannot be reached sooner than the
    #      sailing time from the hub, and the last port's finish time plus the sail home must be <= Tmax.
    m.addConstrs((t[j] >= inst.sail(H, j) * x[H, j] for j in S), name="first_arc")
    m.addConstrs((t[i] + svc[i] + inst.sail(i, H) * x[i, H] <= tmax for i in S), name="last_arc")

    if mode == "delivery":
        # (5) capacitated MTZ: u_i = cargo delivered up to and including port i (tons)
        u = m.addVars(S, lb={i: inst.demand[i] for i in S}, ub=Q, name="u")
        for (i, j) in A:
            if i != H and j != H:
                m.addConstr(u[j] >= u[i] + inst.demand[j] - Q * (1 - x[i, j]), name=f"load[{i},{j}]")
        aux = {"u": u}

    elif mode == "pd":
        # (5') arc flows. D_ij = delivery cargo still on board on arc (i,j),
        #      P_ij = pickup cargo already on board on arc (i,j). Capacity: D + P <= Q * x.
        D = m.addVars(A, lb=0.0, name="D")
        P = m.addVars(A, lb=0.0, name="P")
        m.addConstrs((D[i, j] + P[i, j] <= Q * x[i, j] for (i, j) in A), name="cap")
        # flow conservation: delivery cargo drops by d_i at port i, pickup cargo grows by p_i
        m.addConstrs((D.sum("*", i) - D.sum(i, "*") == inst.demand[i] for i in S), name="dflow")
        m.addConstrs((P.sum(i, "*") - P.sum("*", i) == inst.supply[i] for i in S), name="pflow")
        # ships leave the hub with no pickup cargo and return with no delivery cargo
        m.addConstrs((P[H, j] == 0 for j in S), name="p0")
        m.addConstrs((D[i, H] == 0 for i in S), name="d0")
        aux = {"D": D, "P": P}
    else:
        raise ValueError("mode must be 'delivery' or 'pd'")

    m._x, m._t, m._aux, m._inst, m._svc, m._mode = x, t, aux, inst, svc, mode
    return m


## 4. Solution extraction, verification, and printing

In [ ]:
def extract_routes(m):
    inst, x, svc = m._inst, m._x, m._svc
    H = inst.hub
    nxt = {i: j for (i, j) in x if x[i, j].X > 0.5}
    routes = []
    for j in inst.spokes:
        if x[H, j].X > 0.5:
            r = [H, j]
            while r[-1] != H:
                r.append(nxt[r[-1]])
            dist = sum(inst.dist[a, b] for a, b in zip(r, r[1:]))
            sail = dist / inst.speed
            serv = sum(svc[i] for i in r[1:-1])
            dur = sail + serv
            deliv = sum(inst.demand[i] for i in r[1:-1])
            pick = sum(inst.supply[i] for i in r[1:-1]) if m._mode == "pd" else 0.0
            # peak cargo on board along the route (matters in pickup-and-delivery mode)
            if m._mode == "pd":
                D, P = m._aux["D"], m._aux["P"]
                peak = max(D[a, b].X + P[a, b].X for a, b in zip(r, r[1:]))
            else:
                peak = deliv
            routes.append(dict(route=r, names=" -> ".join(inst.names.get(i, str(i)) for i in r),
                               dist_nm=dist, sail_h=sail, service_h=serv, duration_h=dur,
                               delivery_t=deliv, pickup_t=pick, peak_load_t=peak))
    return routes


def verify(routes, inst, tmax):
    """Independent check of the solver's solution. Returns a list of violations (empty = OK)."""
    bad = []
    visited = [i for r in routes for i in r["route"][1:-1]]
    if sorted(visited) != sorted(inst.spokes):
        bad.append("not every spoke is visited exactly once")
    for r in routes:
        if r["duration_h"] > tmax + 1e-6:
            bad.append(f"duration {r['duration_h']:.1f} h > Tmax on route {r['route']}")
        if r["peak_load_t"] > inst.Q + 1e-6:
            bad.append(f"peak load {r['peak_load_t']:.1f} t > Q on route {r['route']}")
    return bad


def solve(inst, tmax, kmax, mode="delivery", use_service=True, kmin=0,
          time_limit=600, mip_gap=0.0, verbose=False):
    m = build_model(inst, tmax, kmax, mode, use_service, kmin, time_limit, mip_gap, verbose)
    m.optimize()
    status = {GRB.OPTIMAL: "OPTIMAL", GRB.TIME_LIMIT: "TIME_LIMIT",
              GRB.INFEASIBLE: "INFEASIBLE"}.get(m.Status, str(m.Status))
    res = dict(mode=mode, tmax=tmax, kmax=kmax, service=use_service, status=status,
               obj=None, bound=None, gap=None, ships=None, runtime=m.Runtime, routes=[])
    if m.SolCount > 0:
        res["obj"] = m.ObjVal
        res["bound"] = m.ObjBound
        res["gap"] = m.MIPGap
        res["routes"] = extract_routes(m)
        res["ships"] = len(res["routes"])
        res["violations"] = verify(res["routes"], inst, tmax)
    return res


def print_result(res):
    print(f"\n=== mode={res['mode']} | Tmax={res['tmax']} h | K<={res['kmax']} | "
          f"service_time={'yes' if res['service'] else 'no'} ===")
    print(f"status={res['status']}  runtime={res['runtime']:.1f}s")
    if res["obj"] is None:
        print("no feasible solution found within the time limit "
              "(this is NOT proof of infeasibility unless the status is INFEASIBLE)")
        return
    print(f"total distance = {res['obj']:.0f} nm | best bound = {res['bound']:.0f} nm | "
          f"gap = {100*res['gap']:.2f}% | ships used = {res['ships']}")
    for k, r in enumerate(res["routes"], 1):
        print(f"  Ship {k}: {r['names']}")
        print(f"     distance {r['dist_nm']:.0f} nm | sailing {r['sail_h']:.1f} h + port service {r['service_h']:.1f} h "
              f"= {r['duration_h']:.1f} h | delivery {r['delivery_t']:.1f} t | pickup {r['pickup_t']:.1f} t | "
              f"peak load {r['peak_load_t']:.1f} t")
    if res.get("violations"):
        print("  VIOLATIONS:", res["violations"])


## 5. Regression test and sanity checks
A correct model must accept every physically feasible solution. The helper functions below recompute a route's distance and duration **without the solver**, then fix the route in the model and check that it stays feasible. The two reference solutions come from the team's earlier notebooks.

In [ ]:
def route_stats(inst, route, use_service=False):
    """Independent calculation (no solver) of distance, duration and delivery load of one route."""
    dist = sum(inst.dist[a, b] for a, b in zip(route, route[1:]))
    serv = sum(inst.service[i] for i in route[1:-1]) if use_service else 0.0
    return dict(dist_nm=dist, duration_h=dist / inst.speed + serv,
                delivery_t=sum(inst.demand[i] for i in route[1:-1]))


def model_accepts(inst, routes, tmax, kmax, mode="delivery", use_service=False):
    """
    Regression test. Fix the arcs of a known, physically feasible solution and ask the model
    whether it is still feasible. A correct model must accept every feasible solution; a False
    answer means the model is over-restrictive and its 'optimal' results cannot be trusted.
    """
    m = build_model(inst, tmax, kmax, mode, use_service, time_limit=60)
    for r in routes:
        for a, b in zip(r, r[1:]):
            m._x[a, b].LB = 1
    m.optimize()
    return m.Status == GRB.OPTIMAL


## 6. Load the data

In [ ]:
inst = load_instance(EXCEL_PATH)
total_d = sum(inst.demand.values()); total_p = sum(inst.supply.values())
print(f"total demand = {total_d:.1f} t | total supply = {total_p:.1f} t | Q = {inst.Q} t")
print("minimum number of ships by capacity =", math.ceil(max(total_d, total_p) / inst.Q))
pd.DataFrame({"port": [inst.names[i] for i in inst.ports],
              "demand_t": [inst.demand[i] for i in inst.ports],
              "supply_t": [inst.supply[i] for i in inst.ports],
              "service_h": [inst.service[i] for i in inst.ports]}, index=inst.ports).round(2)

### 6b. Run the regression test
Both reference solutions must be accepted (`True`). If any shows `False`, do not trust the optimization results below.

In [ ]:
KNOWN = [
    ("Paper solution, Tmax = 300 h", 300, 4,
     [[1, 9, 4, 5, 1], [1, 2, 1], [1, 12, 11, 8, 7, 6, 15, 1], [1, 16, 14, 13, 3, 10, 1]]),
    ("Earlier notebook solution, Tmax = 350 h", 350, 4,
     [[1, 9, 2, 1], [1, 5, 4, 8, 11, 12, 7, 6, 15, 1], [1, 16, 14, 13, 3, 10, 1]]),
]
for label, T, K, routes in KNOWN:
    stats = [route_stats(inst, r) for r in routes]
    ok_time = all(s["duration_h"] <= T for s in stats)
    ok_cap = all(s["delivery_t"] <= inst.Q for s in stats)
    print(f"{label}: distance = {sum(s['dist_nm'] for s in stats):.0f} nm | "
          f"longest route = {max(s['duration_h'] for s in stats):.1f} h | "
          f"feasible by hand: {ok_time and ok_cap} | accepted by model: {model_accepts(inst, routes, T, K)}")

## 7. Scenario A: replicate the paper (Tmax = 300 h, no service time)
The old code stopped with a 7.48% gap after 600 s. With the two-index formulation the solver should prove optimality, typically within a few minutes.

In [ ]:
resA = solve(inst, tmax=300, kmax=4, mode="delivery", use_service=False, time_limit=TIME_LIMIT)
print_result(resA)

## 8. Scenario B: with port service time (Sheet2)
Once loading and unloading time is counted, the Scenario A solution becomes **infeasible**. The fleet is opened up to 6 ships so that the problem stays feasible.

In [ ]:
resB = solve(inst, tmax=300, kmax=6, mode="delivery", use_service=True, time_limit=TIME_LIMIT)
print_result(resB)

## 9. Scenario C: Tmax = 350 h, no service time
The earlier notebook found 9,708 nm. If this run stops at TIME_LIMIT, raise `TIME_LIMIT` (for example to 3600).

In [ ]:
resC = solve(inst, tmax=350, kmax=4, mode="delivery", use_service=False, time_limit=TIME_LIMIT)
print_result(resC)

## 10. Scenario D: simultaneous pickup and delivery
Uses the *Supply* column. Continuous flow variables make optimality proofs slower; report the *gap* if the run stops at the time limit.

In [ ]:
resD = solve(inst, tmax=300, kmax=4, mode="pd", use_service=False, time_limit=TIME_LIMIT)
print_result(resD)

## 11. Route map (optional)
Coordinates come from the team's map notebook. Lines are straight segments between ports, not actual sea lanes.

In [ ]:
COORDS = {1:(-7.206,112.733), 2:(-1.265,116.831), 3:(-3.328,114.592), 4:(3.326,117.578),
          5:(-0.502,117.145), 6:(-0.891,119.870), 7:(0.536,123.056), 8:(-0.951,122.787),
          9:(2.133,117.501), 10:(-2.534,112.944), 11:(-3.974,122.515), 12:(-5.460,122.624),
          13:(-1.014,100.367), 14:(0.533,101.450), 15:(-3.300,116.598), 16:(1.130,104.030)}

def draw_map(res, filename="ship_routes.html"):
    import folium
    colors = ["red", "blue", "orange", "purple", "green", "black"]
    m = folium.Map(location=[-2, 112], zoom_start=5)
    for i, (lat, lon) in COORDS.items():
        folium.Marker([lat, lon], tooltip=f"{i} - {inst.names[i]}",
                      icon=folium.Icon(color="blue" if i == 1 else "green")).add_to(m)
    for k, r in enumerate(res["routes"]):
        folium.PolyLine([COORDS[i] for i in r["route"]], color=colors[k % len(colors)],
                        weight=4, opacity=0.8,
                        tooltip=f"Ship {k+1}: {r['dist_nm']:.0f} nm, {r['duration_h']:.0f} h").add_to(m)
    m.save(filename)
    return m

draw_map(resB)   # use resA / resC / resD for the other scenarios

## 12. Tmax sensitivity analysis (for the Results section of the paper)
Produces a table of total distance and number of ships for several `Tmax` values, with and without service time. It takes a long time; enable it with `RUN_SWEEP = True`.

In [ ]:
def sweep(inst, tmax_values, kmax, modes=("delivery",), services=(False, True), time_limit=300):
    rows = []
    for mode in modes:
        for sv in services:
            for T in tmax_values:
                r = solve(inst, T, kmax, mode, sv, time_limit=time_limit)
                rows.append(dict(mode=mode, service=sv, tmax=T, status=r["status"],
                                 ships=r["ships"], dist_nm=r["obj"], bound_nm=r["bound"],
                                 gap_pct=None if r["gap"] is None else round(100 * r["gap"], 2)))
                print(rows[-1], flush=True)
    df = pd.DataFrame(rows)
    df.to_csv("sensitivity_tmax.csv", index=False)
    return df


In [ ]:
if RUN_SWEEP:
    df_sweep = sweep(inst, [250, 300, 350, 400, 500, 10_000], kmax=6,
                     modes=("delivery",), services=(False, True), time_limit=TIME_LIMIT)
    display(df_sweep)
else:
    print("RUN_SWEEP = False. Set it to True in the settings cell to run the sweep.")

## Interpretation notes
* Status `OPTIMAL` with a 0% gap is the only basis for writing "optimal" in the paper. A `TIME_LIMIT` status must be reported as a *best-found solution* together with its gap.
* `TIME_LIMIT` without a solution is not proof of infeasibility; only `INFEASIBLE` proves it.
* The triangle-inequality warning in Section 6 means some distances must be verified against an official distance table before any result is reported.
* Demand and supply need an explicit time basis (per week, per voyage, and so on) to be consistent with `Tmax` and ship capacity.
* If the model's optimum is ever worse than a solution you can construct by hand, treat that as a modelling error and rerun Section 6b.